# 01 · COPY INTO: Catálogo y Sucursales → Bronze
Tarea 1 del job. Carga idempotente: COPY INTO recuerda qué archivos ya cargó, así que volver a correrlo
no duplica filas. Corre fuera del pipeline porque COPY INTO no se puede declarar en un Lakeflow Spark
Declarative Pipeline; el pipeline lee después estas tablas en batch desde una Materialized View.

**Por qué COPY INTO en estas dos fuentes:** llegan como archivos completos y pocos (2 snapshots del catálogo,
1 archivo de sucursales), sin necesidad de evolución de esquema avanzada. Auto Loader queda para las fuentes
que llegan por lotes incrementales.

In [ ]:
dbutils.widgets.text("catalog", "electrocasa_dev", "Catálogo del proyecto")
catalogo = dbutils.widgets.get("catalog")
print("Catálogo:", catalogo)

In [ ]:
sql = """
COPY INTO ${electrocasa.catalog}.bronze.catalogo
FROM (
  SELECT
    producto_id, nombre_producto, categoria, marca, precio_lista, activo,
    _metadata.file_name AS _archivo_origen,
    regexp_extract(_metadata.file_name, '([0-9]{4}-[0-9]{2}-[0-9]{2})', 1) AS _lote,
    current_timestamp() AS _fecha_ingesta
  FROM '/Volumes/${electrocasa.catalog}/default/vol_landing/catalogo/'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'true', 'primitivesAsString' = 'true');
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
COPY INTO ${electrocasa.catalog}.bronze.sucursales
FROM (
  SELECT
    sucursal_id, nombre_sucursal, distrito, ciudad, region, fecha_apertura,
    _metadata.file_name AS _archivo_origen,
    'referencia' AS _lote,
    current_timestamp() AS _fecha_ingesta
  FROM '/Volumes/${electrocasa.catalog}/default/vol_landing/sucursales/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- Evidencia de idempotencia: cada COPY INTO deja una versión con numCopiedFiles / numOutputRows.
-- Una segunda corrida sin archivos nuevos deja numCopiedFiles = 0.
SELECT version, timestamp, operation, operationMetrics
FROM (DESCRIBE HISTORY ${electrocasa.catalog}.bronze.catalogo)
ORDER BY version DESC LIMIT 5;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
SELECT _archivo_origen, count(*) AS filas
FROM ${electrocasa.catalog}.bronze.catalogo GROUP BY _archivo_origen
UNION ALL
SELECT _archivo_origen, count(*) FROM ${electrocasa.catalog}.bronze.sucursales GROUP BY _archivo_origen;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)